In [ ]:
import os

import pandas as pd
import requests
from bs4 import BeautifulSoup

from src.constants import WORLD_CUP_YEARS
from src.utils import get_project_root

In [ ]:
root = get_project_root()

In [ ]:
def get_matches(year):
    headers = {"User-Agent": "Mozilla/5.0"}
    web_url = f'https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup'

    response = requests.get(url=web_url, headers=headers)
    content = response.text
    soup = BeautifulSoup(content, 'lxml')

    home = []
    score = []
    away = []

    if year >= 1998:
        group_stage_matches = soup.find_all("tr", style="font-size:90%")
        for match_ in group_stage_matches:
            tds = match_.find_all("td", recursive=False)
            if len(tds) == 4:
                home.append(tds[0].find("a").get_text(strip=True))
                score.append(tds[1].find("a").get_text(strip=True))
                away.append(tds[2].find("a").get_text(strip=True))

    matches = soup.find_all('div', class_ = 'footballbox')
    for match_ in matches:
        home.append(match_.find('th', class_='fhome').get_text().replace("\xa0", ""))
        score.append(match_.find('th', class_='fscore').get_text().replace("\xa0", ""))
        away.append(match_.find('th', class_='faway').get_text().replace("\xa0", ""))

    dict_football = {'home': home, 'score': score, 'away': away}
    df_football = pd.DataFrame(dict_football)
    df_football['year'] = year
    return df_football

In [ ]:
fifa = [get_matches(year) for year in WORLD_CUP_YEARS]
df_fifa = pd.concat(fifa, ignore_index=True)

In [ ]:
output_dir = f"{root}/data/raw/historical"
os.makedirs(output_dir, exist_ok=True)

df_fifa.to_csv(f"{root}/data/raw/historical/fifa_world_cup_historical_data.csv", index=False)

In [ ]:
output_dir = f"{root}/data/raw/fixture"
os.makedirs(output_dir, exist_ok=True)

df_fixture = get_matches(2026)
df_fixture.to_csv(f"{root}/data/raw/fixture/fifa_world_cup_fixture_data.csv", index=False)